# TD-MPC2 vs SAC on highway-env

Group 13 | MLR 555 | Goal: verify environment, GPU, dependencies, and that highway-env, SB3, and TD-MPC2 all load cleanly before touching RL code.

**Exit criteria:** all `[ok]` checks pass at the end. Continuous-action highway-env works. TD-MPC2 imports.

In [1]:
# Cell 1 — Mount Drive + pin project root (idempotent)
from google.colab import drive
drive.mount('/content/drive')

import os, sys
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/tdmpc2-highway')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Working dir: {os.getcwd()}")
print(f"sys.path[0]: {sys.path[0]}")

Mounted at /content/drive
Working dir: /content/drive/MyDrive/tdmpc2-highway
sys.path[0]: /content/drive/MyDrive/tdmpc2-highway


In [2]:
# Cell 2 — Confirm GPU. Note which one we're on so I can tune budgets.
!nvidia-smi

Mon May  4 17:10:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   29C    P0             70W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
# Cell 3 — System packages: ffmpeg for demo videos (Day 3),
# swig for any Box2D-based gym envs we might fall back on.
!apt-get install -y -qq ffmpeg swig > /dev/null 2>&1
print("[ok] system deps installed")

[ok] system deps installed


In [4]:
# Cell 4 — Python deps. numpy<2 pin: SB3 + several RL libs still trip
# on the NumPy 2.x C-ABI break as of early 2026.
%%capture install_log
!pip install -q \
    "numpy<2" \
    "torch>=2.1" \
    "gymnasium>=0.29" \
    "highway-env>=1.8" \
    "stable-baselines3[extra]>=2.2" \
    "hydra-core>=1.3" \
    "omegaconf>=2.3" \
    "tensordict>=0.3" \
    termcolor wandb moviepy imageio imageio-ffmpeg \
    matplotlib pandas seaborn tqdm pyyaml

In [5]:
print("Install complete. If anything breaks downstream, run: install_log.show()")

Install complete. If anything breaks downstream, run: install_log.show()


### ⚠️ Restart runtime now

The `numpy<2` pin requires a kernel restart. Go to **Runtime → Restart session**, then re-run Cell 1 (Drive mount) and continue from Cell 6. Do **not** re-run Cell 4 — packages persist.

In [6]:
# Cell 6 — Verify NumPy pin + GPU survived the restart.
import numpy as np
import torch

print(f"NumPy: {np.__version__}")
print(f"Torch: {torch.__version__}")
print(f"CUDA:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:   {torch.cuda.get_device_name(0)}")
    print(f"VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

assert np.__version__.startswith("1."), f"NumPy {np.__version__} — pin failed, re-run Cell 4 + restart"
assert torch.cuda.is_available(), "No GPU"
print("\n[ok] post-restart environment good")

NumPy: 1.26.4
Torch: 2.10.0+cu128
CUDA:  True
GPU:   NVIDIA H100 80GB HBM3
VRAM:  85.0 GB

[ok] post-restart environment good


In [7]:
# Cell 7 — Create the src/ package layout. Both training notebooks
# import from these modules, so the structure must exist before we
# write the modules themselves.
import os
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/tdmpc2-highway")
for sub in [
    "src", "src/envs", "src/training", "src/evaluation", "src/utils",
    "configs", "scripts", "notebooks", "results", "videos",
    "checkpoints", "logs", "third_party",
]:
    (ROOT / sub).mkdir(parents=True, exist_ok=True)

# Mark every src/* directory as a Python package
for pkg in ["src", "src/envs", "src/training", "src/evaluation", "src/utils"]:
    init = ROOT / pkg / "__init__.py"
    init.touch(exist_ok=True)

# Make src/ importable from notebooks
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("[ok] structure created")
!ls -la

[ok] structure created
total 36
drwx------ 2 root root 4096 May  4 16:58 checkpoints
drwx------ 2 root root 4096 May  4 16:58 configs
drwx------ 2 root root 4096 May  4 16:58 logs
drwx------ 2 root root 4096 May  4 16:58 notebooks
drwx------ 2 root root 4096 May  4 16:58 results
drwx------ 2 root root 4096 May  4 16:58 scripts
drwx------ 6 root root 4096 May  4 16:58 src
drwx------ 2 root root 4096 May  4 16:58 third_party
drwx------ 2 root root 4096 May  4 16:58 videos


In [8]:
# Cell 8 — Central config: every script reads from here so SAC and
# TD-MPC2 train under identical env settings.
config_py = '''"""
Central configuration for the TD-MPC2 vs SAC highway-env study.
Single source of truth: every script imports from here so SAC, TD-MPC2,
and zero-shot evaluation share identical environment settings.
"""
from pathlib import Path

PROJECT_ROOT    = Path("/content/drive/MyDrive/tdmpc2-highway")
RESULTS_DIR     = PROJECT_ROOT / "results"
VIDEOS_DIR      = PROJECT_ROOT / "videos"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints"
LOGS_DIR        = PROJECT_ROOT / "logs"
for d in [RESULTS_DIR, VIDEOS_DIR, CHECKPOINTS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Identical config for training (highway-v0) and zero-shot eval
# (merge-v0, roundabout-v0). Shared config = fair transfer comparison.
HIGHWAY_CONFIG = {
    "action": {
        "type": "ContinuousAction",
        "longitudinal": True,
        "lateral": True,
    },
    "observation": {
        "type": "Kinematics",
        "vehicles_count": 5,                # ego + 4 nearest
        "features": ["presence", "x", "y", "vx", "vy"],
        "absolute": False,                  # ego-relative
        "normalize": True,
        "order": "sorted",
    },
    "simulation_frequency": 15,
    "policy_frequency": 5,
    "duration": 40,
    "vehicles_count": 20,
    "lanes_count": 4,
    "collision_reward": -1.0,
    "right_lane_reward": 0.1,
    "high_speed_reward": 0.4,
    "lane_change_reward": 0.0,
    "reward_speed_range": [20, 30],
    "normalize_reward": True,
    "offroad_terminal": True,
}

TRAIN_ENV_ID = "highway-v0"
EVAL_ENV_IDS = ["highway-v0", "merge-v0", "roundabout-v0"]

# Training budgets (3-day project; multi-seed shares this budget)
TOTAL_TIMESTEPS  = 100_000
EVAL_INTERVAL    = 5_000
SEEDS            = [0, 1, 2]
N_EVAL_EPISODES  = 30

# wandb
WANDB_PROJECT = "tdmpc2-highway"
WANDB_ENTITY  = None    # override in notebook if you want a team
'''

(ROOT / "src/utils/config.py").write_text(config_py)
print("[ok] src/utils/config.py written")

[ok] src/utils/config.py written


In [9]:
# Cell 9 (REPLACE) — Env factory with per-env action-space selection.
# Continuous on highway-v0 (training); discrete on merge/roundabout
# (zero-shot eval) because their reward functions assume discrete actions.
factory_py = '''"""
Highway-env factory + wrappers.

Action-space policy:
  - highway-v0:    continuous (training env for both SAC and TD-MPC2)
  - merge-v0:      discrete (its _rewards() hard-codes `action in [0,2]`)
  - roundabout-v0: discrete (same upstream constraint)

Zero-shot transfer is therefore *cross-action-representation*: we
discretize the trained continuous policy at eval time. This is more
novel than same-action transfer and is the standard configuration
for these envs in the highway-env literature.
"""
from __future__ import annotations
import copy
from typing import Optional

import gymnasium as gym
import highway_env  # noqa: F401  registers env IDs
import numpy as np

from src.utils.config import HIGHWAY_CONFIG


# Per-env action type. Override in HIGHWAY_CONFIG by passing config_overrides.
_ENV_ACTION_TYPE = {
    "highway-v0":    "ContinuousAction",
    "merge-v0":      "DiscreteMetaAction",
    "roundabout-v0": "DiscreteMetaAction",
}


def _build_config(env_id: str, overrides: Optional[dict]) -> dict:
    cfg = copy.deepcopy(HIGHWAY_CONFIG)
    action_type = _ENV_ACTION_TYPE.get(env_id, "ContinuousAction")
    if action_type == "ContinuousAction":
        cfg["action"] = {
            "type": "ContinuousAction",
            "longitudinal": True,
            "lateral": True,
        }
    else:
        cfg["action"] = {"type": "DiscreteMetaAction"}
    if overrides:
        cfg.update(overrides)
    return cfg


class FlattenObservation(gym.ObservationWrapper):
    """Flatten (V, F) kinematics matrix into a 1-D vector for MLP policies."""
    def __init__(self, env: gym.Env):
        super().__init__(env)
        low  = self.observation_space.low.flatten()
        high = self.observation_space.high.flatten()
        self.observation_space = gym.spaces.Box(low=low, high=high, dtype=np.float32)

    def observation(self, obs: np.ndarray) -> np.ndarray:
        return obs.astype(np.float32).flatten()


class MetricsWrapper(gym.Wrapper):
    """Track per-episode collision and success."""
    def __init__(self, env: gym.Env):
        super().__init__(env)
        self._collided = False
        self._steps = 0

    def reset(self, **kwargs):
        self._collided = False
        self._steps = 0
        return self.env.reset(**kwargs)

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self._steps += 1
        if info.get("crashed", False):
            self._collided = True
        if terminated or truncated:
            info["episode_collision"] = bool(self._collided)
            info["episode_success"]   = bool((not self._collided) and truncated)
            info["episode_length"]    = int(self._steps)
        return obs, reward, terminated, truncated, info


def make_highway_env(
    env_id: str = "highway-v0",
    seed: Optional[int] = None,
    render_mode: Optional[str] = None,
    config_overrides: Optional[dict] = None,
) -> gym.Env:
    """
    env_id:   "highway-v0" | "merge-v0" | "roundabout-v0"
    seed:     deterministic per (env_id, seed)
    render_mode: "rgb_array" for demo videos
    """
    cfg = _build_config(env_id, config_overrides)
    env = gym.make(env_id, config=cfg, render_mode=render_mode)
    env = FlattenObservation(env)
    env = MetricsWrapper(env)

    if seed is not None:
        env.reset(seed=seed)
        env.action_space.seed(seed)

    return env


def get_env_action_type(env_id: str) -> str:
    """Public helper: returns 'continuous' or 'discrete' for a given env_id."""
    return "continuous" if _ENV_ACTION_TYPE.get(env_id) == "ContinuousAction" else "discrete"
'''

(ROOT / "src/envs/highway_factory.py").write_text(factory_py)
print("[ok] src/envs/highway_factory.py updated (per-env action types)")

[ok] src/envs/highway_factory.py updated (per-env action types)


In [10]:
# Cell 10 (REPLACE) — Verify the three envs load with their respective
# action types: continuous for highway-v0, discrete for merge/roundabout.
import importlib
import src.envs.highway_factory as hf
importlib.reload(hf)
from src.envs.highway_factory import make_highway_env, get_env_action_type
import gymnasium as gym

for env_id in ["highway-v0", "merge-v0", "roundabout-v0"]:
    env = make_highway_env(env_id=env_id, seed=0)
    obs, info = env.reset(seed=0)
    assert obs.ndim == 1, f"{env_id} obs not flat: {obs.shape}"

    a_type = get_env_action_type(env_id)
    if a_type == "continuous":
        assert isinstance(env.action_space, gym.spaces.Box), f"{env_id} should be Box"
    else:
        assert isinstance(env.action_space, gym.spaces.Discrete), f"{env_id} should be Discrete"

    # One full episode to confirm safety metrics surface on terminal step
    seen_metrics = False
    for _ in range(400):
        a = env.action_space.sample()
        o, r, term, trunc, info = env.step(a)
        if term or trunc:
            assert "episode_collision" in info
            assert "episode_success"   in info
            seen_metrics = True
            break
    assert seen_metrics, f"{env_id} never terminated in 400 steps"
    print(f"[ok] {env_id}: action_type={a_type}, obs={obs.shape}, action_space={env.action_space}")
    env.close()

print("\n[ok] all three envs functional")

[ok] highway-v0: action_type=continuous, obs=(25,), action_space=Box(-1.0, 1.0, (2,), float32)
[ok] merge-v0: action_type=discrete, obs=(25,), action_space=Discrete(5)
[ok] roundabout-v0: action_type=discrete, obs=(25,), action_space=Discrete(5)

[ok] all three envs functional


In [11]:
# Cell 11 — wandb login. Paste your key from https://wandb.ai/authorize
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: error1rak (error1rak-american-university-of-sharjah) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [12]:
# Cell 12 — Custom SB3 callback: log per-episode collision/success
# rates to wandb. Standard EvalCallback only logs reward; we need
# safety-critical metrics absent from the original TD-MPC2 paper.
callbacks_py = '''"""
Custom SB3 callback: log per-episode collision and success rates.
Pushed to wandb at every rollout end so paper plots can use them directly.
"""
from __future__ import annotations
import numpy as np
import wandb
from stable_baselines3.common.callbacks import BaseCallback


class SafetyMetricsCallback(BaseCallback):
    """Log collision rate, success rate, and episode length to wandb."""

    def __init__(self, verbose: int = 0):
        super().__init__(verbose)
        self._collisions: list[bool] = []
        self._successes:  list[bool] = []
        self._lengths:    list[int]  = []

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            if "episode_collision" in info:
                self._collisions.append(info["episode_collision"])
                self._successes.append(info["episode_success"])
                self._lengths.append(info["episode_length"])
        return True

    def _on_rollout_end(self) -> None:
        if not self._collisions:
            return
        metrics = {
            "metrics/collision_rate": float(np.mean(self._collisions)),
            "metrics/success_rate":   float(np.mean(self._successes)),
            "metrics/episode_length": float(np.mean(self._lengths)),
            "metrics/n_episodes":     len(self._collisions),
            "global_step":            self.num_timesteps,
        }
        if wandb.run is not None:
            wandb.log(metrics)
        if self.verbose:
            print(f"[step {self.num_timesteps}] {metrics}")
        self._collisions.clear()
        self._successes.clear()
        self._lengths.clear()
'''

(ROOT / "src/training/callbacks.py").write_text(callbacks_py)
print("[ok] src/training/callbacks.py written")

[ok] src/training/callbacks.py written


In [13]:
# Cell 13 — SAC training module. One function = one seed = one wandb
# run. The notebook driver loops over SEEDS to produce 3 runs that the
# plotting code aggregates into 95% CI curves.
train_sac_py = '''"""
SAC baseline for highway-v0 (model-free).

train_sac(seed=N) trains one seed end-to-end. Loop over SEEDS in the
notebook driver to produce the multi-seed comparison.
"""
from __future__ import annotations
import argparse
from pathlib import Path

import numpy as np
import torch
import wandb
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import CallbackList, EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from wandb.integration.sb3 import WandbCallback

from src.envs.highway_factory import make_highway_env
from src.training.callbacks import SafetyMetricsCallback
from src.utils.config import (
    CHECKPOINTS_DIR, EVAL_INTERVAL, TOTAL_TIMESTEPS,
    TRAIN_ENV_ID, WANDB_ENTITY, WANDB_PROJECT,
)


def _make_vec_env(env_id: str, seed: int, monitor_dir: Path) -> DummyVecEnv:
    """SB3 needs a VecEnv; we wrap a single env with Monitor for episode logging."""
    def _thunk():
        env = make_highway_env(env_id=env_id, seed=seed)
        return Monitor(env, filename=str(monitor_dir / f"sac_seed{seed}"))
    return DummyVecEnv([_thunk])


def train_sac(
    seed: int,
    total_timesteps: int = TOTAL_TIMESTEPS,
    env_id: str = TRAIN_ENV_ID,
    eval_interval: int = EVAL_INTERVAL,
    use_wandb: bool = True,
) -> Path:
    """Train one SAC seed; return path to the saved final model."""
    np.random.seed(seed)
    torch.manual_seed(seed)

    run_name = f"sac_{env_id}_seed{seed}"
    ckpt_dir = CHECKPOINTS_DIR / run_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    if use_wandb:
        wandb.init(
            project=WANDB_PROJECT, entity=WANDB_ENTITY, name=run_name,
            config={"algo": "SAC", "env_id": env_id, "seed": seed,
                    "total_timesteps": total_timesteps},
            sync_tensorboard=True, reinit=True,
        )

    train_env = _make_vec_env(env_id, seed=seed,        monitor_dir=ckpt_dir)
    eval_env  = _make_vec_env(env_id, seed=seed + 1000, monitor_dir=ckpt_dir)

    # Hyperparameters: SB3 defaults work for highway-env. learning_starts
    # avoids premature updates from a near-empty buffer; gamma=0.95 reflects
    # episodes capped at 200 steps.
    model = SAC(
        policy="MlpPolicy",
        env=train_env,
        learning_rate=3e-4,
        buffer_size=100_000,
        learning_starts=1_000,
        batch_size=256,
        tau=0.005,
        gamma=0.95,
        train_freq=1,
        gradient_steps=1,
        ent_coef="auto",
        verbose=1,
        seed=seed,
        device="cuda" if torch.cuda.is_available() else "cpu",
        tensorboard_log=str(ckpt_dir / "tb"),
    )

    callbacks = [
        SafetyMetricsCallback(verbose=0),
        EvalCallback(
            eval_env,
            best_model_save_path=str(ckpt_dir / "best"),
            log_path=str(ckpt_dir / "eval"),
            eval_freq=eval_interval,
            n_eval_episodes=10,
            deterministic=True,
            render=False,
        ),
    ]
    if use_wandb:
        callbacks.append(WandbCallback(
            gradient_save_freq=0,
            model_save_path=str(ckpt_dir / "wandb_models"),
            verbose=0,
        ))

    model.learn(
        total_timesteps=total_timesteps,
        callback=CallbackList(callbacks),
        progress_bar=True,
    )

    final_path = ckpt_dir / "final.zip"
    model.save(str(final_path))
    print(f"[SAC seed={seed}] saved final → {final_path}")

    train_env.close(); eval_env.close()
    if use_wandb and wandb.run is not None:
        wandb.finish()

    return final_path


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--seed", type=int, required=True)
    parser.add_argument("--steps", type=int, default=TOTAL_TIMESTEPS)
    parser.add_argument("--no-wandb", action="store_true")
    args = parser.parse_args()
    train_sac(
        seed=args.seed,
        total_timesteps=args.steps,
        use_wandb=not args.no_wandb,
    )
'''

(ROOT / "src/training/train_sac.py").write_text(train_sac_py)
print("[ok] src/training/train_sac.py written")

[ok] src/training/train_sac.py written


In [14]:
# Cell 14 — Tiny SAC run (2k steps, no wandb) to confirm the pipeline
# end-to-end before committing to a 30+ minute full training run.
import importlib
import src.training.train_sac
importlib.reload(src.training.train_sac)
from src.training.train_sac import train_sac

_ = train_sac(seed=99, total_timesteps=2_000, use_wandb=False)
print("\n[ok] SAC pipeline functional end-to-end")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using cuda device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Logging to /content/drive/MyDrive/tdmpc2-highway/checkpoints/sac_highway-v0_seed99/tb/SAC_1


Output()

...

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

[SAC seed=99] saved final → /content/drive/MyDrive/tdmpc2-highway/checkpoints/sac_highway-v0_seed99/final.zip

[ok] SAC pipeline functional end-to-end


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [17]:
# T1 — Clone Hansen et al.'s official TD-MPC2 repo into third_party/.
# We don't pip-install it; we add its package dir to sys.path so we
# can instantiate the TDMPC2 agent class in our own training loop
# without fighting Hydra's CLI scaffolding.
import os
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/tdmpc2-highway")
tdmpc2_repo = ROOT / "third_party" / "tdmpc2"

if not tdmpc2_repo.exists():
    os.makedirs(ROOT / "third_party", exist_ok=True)
    !cd {ROOT}/third_party && git clone --depth=1 https://github.com/nicklashansen/tdmpc2.git
else:
    print("[ok] tdmpc2 repo already cloned")
    !cd {tdmpc2_repo} && git log -1 --oneline

# The actual Python package lives at third_party/tdmpc2/tdmpc2/
print("\nPackage contents:")
!ls {tdmpc2_repo}/tdmpc2/ 2>/dev/null | head -20

Output hidden; open in https://colab.research.google.com to view.

In [18]:
# T2 — TD-MPC2 needs a couple of packages our requirements didn't pull in.
# Quiet install; should take ~30s.
!pip install -q pyrallis colorednoise mujoco
print("[ok] tdmpc2 extra deps installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 32.6 MB/s eta 0:00:00
[ok] tdmpc2 extra deps installed


In [24]:
# T2.5 (FIXED) — Install latest torchrl. Older versions (<0.5) have
# been yanked from PyPI; the API surface used by TD-MPC2's Buffer
# (ReplayBuffer, LazyTensorStorage, SliceSampler) is stable across
# 0.5+ releases.
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "torchrl"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("INSTALL FAILED:")
    print(result.stderr)
    raise SystemExit(1)
print("[ok] torchrl installed")

# Verify import (this will hard-fail if install didn't actually work)
import torchrl
from torchrl.data.replay_buffers import ReplayBuffer, LazyTensorStorage
from torchrl.data.replay_buffers.samplers import SliceSampler
print(f"[ok] torchrl {torchrl.__version__} importable")

[ok] torchrl installed
[ok] torchrl 0.12.0 importable


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [29]:
# T3 (REPLACE) — Adapter aligned with TD-MPC2's TensorWrapper contract.
# Key fixes:
#   - obs_shape is a dict {"state": (D,)} not a list
#   - step() returns (obs, reward_tensor, done, info) with info['success']
#     and info['terminated'] (tensor, required by their trainer)
#   - reset() returns just obs (no info tuple)
adapter_py = '''"""
Adapter: highway-env -> TD-MPC2 task interface.

Matches the contract of tdmpc2/envs/wrappers/tensor.py exactly so the
official TDMPC2 agent and Buffer can consume our env unchanged.
"""
from __future__ import annotations
from collections import defaultdict
from typing import Any

import gymnasium as gym
import numpy as np
import torch

from src.envs.highway_factory import make_highway_env


class HighwayTDMPC2Wrapper(gym.Wrapper):
    """Highway-env -> TD-MPC2 TensorWrapper-compatible env."""

    def __init__(self, env: gym.Env, max_episode_steps: int = 200):
        super().__init__(env)
        self.max_episode_steps = max_episode_steps

    def rand_act(self) -> torch.Tensor:
        return torch.from_numpy(self.action_space.sample().astype(np.float32))

    def _to_tensor(self, x):
        if isinstance(x, np.ndarray):
            x = torch.from_numpy(x)
            if x.dtype == torch.float64:
                x = x.float()
        return x

    def reset(self, **kwargs) -> torch.Tensor:
        obs, _info = self.env.reset(**kwargs)
        return self._to_tensor(obs.astype(np.float32))

    def step(self, action: Any):
        if isinstance(action, torch.Tensor):
            action = action.detach().cpu().numpy().astype(np.float32)

        obs, reward, terminated, truncated, info = self.env.step(action)
        done = bool(terminated or truncated)

        # Required by TD-MPC2 trainer/buffer:
        #   info['success']    : float (1.0 = successful episode)
        #   info['terminated'] : tensor (True = real termination, e.g. crash;
        #                                False = time-out truncation)
        info = defaultdict(float, info)
        info["success"]    = float(info.get("episode_success", False))
        info["terminated"] = torch.tensor(float(terminated))

        return (
            self._to_tensor(obs.astype(np.float32)),
            torch.tensor(reward, dtype=torch.float32),
            done,
            info,
        )


def make_tdmpc2_highway_env(
    env_id: str = "highway-v0",
    seed: int = 0,
    max_episode_steps: int = 200,
):
    base = make_highway_env(env_id=env_id, seed=seed)
    return HighwayTDMPC2Wrapper(base, max_episode_steps=max_episode_steps)
'''

(ROOT / "src/envs/tdmpc2_adapter.py").write_text(adapter_py)
print("[ok] src/envs/tdmpc2_adapter.py rewritten (TensorWrapper-compatible)")

[ok] src/envs/tdmpc2_adapter.py rewritten (TensorWrapper-compatible)


In [30]:
# T4 (REPLACE) — TD-MPC2 training driver that mirrors the official
# OnlineTrainer.train() loop line-by-line, but with our wandb logging
# instead of their Logger class. Builds the cfg the way their parse_cfg
# does (with correct dict-typed obs_shape, derived bin_size, model_size
# expansion, etc.) so the agent/buffer accept it unchanged.
train_tdmpc2_py = '''"""
TD-MPC2 training for highway-v0.

Runs an online-RL loop that mirrors the official OnlineTrainer line-by-line:
  - Random actions for the first cfg.seed_steps environment steps
  - Planned actions thereafter (MPPI inside agent.act)
  - Per-episode TensorDict accumulated, pushed into Buffer at episode end
  - Agent updated every step after seed_steps, with a warmup at exactly
    step == seed_steps (seed_steps updates in a row)
"""
from __future__ import annotations
import sys
import time
from pathlib import Path

import numpy as np
import torch
import wandb
from omegaconf import OmegaConf
from tensordict.tensordict import TensorDict

# Make the official tdmpc2 package importable
TDMPC2_REPO = Path("/content/drive/MyDrive/tdmpc2-highway/third_party/tdmpc2/tdmpc2")
if str(TDMPC2_REPO) not in sys.path:
    sys.path.insert(0, str(TDMPC2_REPO))

from tdmpc2 import TDMPC2
from common.buffer import Buffer
from common import MODEL_SIZE

from src.envs.tdmpc2_adapter import make_tdmpc2_highway_env
from src.utils.config import (
    CHECKPOINTS_DIR, TOTAL_TIMESTEPS, TRAIN_ENV_ID,
    WANDB_ENTITY, WANDB_PROJECT,
)


def _build_cfg(env, seed: int, total_timesteps: int, model_size: int = 1):
    """
    Build a TD-MPC2 cfg dict that matches what the official parse_cfg
    produces. We intentionally avoid Hydra so this works inside notebooks.
    """
    obs_dim = int(np.prod(env.observation_space.shape))
    act_dim = int(np.prod(env.action_space.shape))
    ep_len  = env.max_episode_steps

    cfg = {
        # Task / env
        "task": "highway-v0",
        "obs": "state",
        "obs_shape": {"state": (obs_dim,)},        # dict, not list
        "action_dim": act_dim,
        "episode_length": ep_len,
        "episode_lengths": [ep_len],               # plural form, used by agent
        "seed_steps": max(1000, 5 * ep_len),
        "episodic": True,                          # highway terminates on crash
        "multitask": False,
        "tasks": ["highway-v0"],
        "task_dim": 0,
        # Training budget
        "steps": total_timesteps,
        "seed": seed,
        # Optimization
        "batch_size": 256,
        "lr": 3e-4,
        "enc_lr_scale": 0.3,
        "grad_clip_norm": 20.0,
        "tau": 0.01,
        "rho": 0.5,
        "consistency_coef": 20.0,
        "reward_coef": 0.1,
        "value_coef": 0.1,
        "termination_coef": 1.0,
        # Discount: derived per parse_cfg from episode length
        "discount_denom": 5,
        "discount_min": 0.95,
        "discount_max": 0.995,
        # Buffer
        "buffer_size": min(100_000, total_timesteps),
        # Planning (MPPI)
        "mpc": True,
        "horizon": 3,
        "iterations": 6,
        "num_samples": 512,
        "num_elites": 64,
        "num_pi_trajs": 24,
        "min_std": 0.05,
        "max_std": 2.0,
        "temperature": 0.5,
        # Actor
        "log_std_min": -10,
        "log_std_max": 2,
        "entropy_coef": 1e-4,
        # Critic (distributional regression)
        "num_bins": 101,
        "vmin": -10.0,
        "vmax": 10.0,
        # Architecture (set via MODEL_SIZE below)
        "model_size": model_size,
        "num_channels": 32,
        "dropout": 0.01,
        "simnorm_dim": 8,
        # Misc — disable compile (Colab + small batch ⇒ compile overhead dominates)
        "compile": False,
        "save_video": False,
        "save_agent": False,
        "enable_wandb": False,                     # we log to wandb manually
        "wandb_silent": True,
        "save_csv": False,
    }

    # Derived: bin_size for distributional regression (parse_cfg formula)
    cfg["bin_size"] = (cfg["vmax"] - cfg["vmin"]) / (cfg["num_bins"] - 1)

    # Model-size expansion (parse_cfg copies MODEL_SIZE[k] into cfg)
    for k, v in MODEL_SIZE[model_size].items():
        cfg[k] = v

    return OmegaConf.create(cfg)


def _to_td(obs, env, action=None, reward=None, terminated=None):
    """Build a per-step TensorDict (mirrors OnlineTrainer.to_td)."""
    if isinstance(obs, dict):
        obs = TensorDict(obs, batch_size=(), device="cpu")
    else:
        obs = obs.unsqueeze(0).cpu()
    if action is None:
        action = torch.full_like(env.rand_act(), float("nan"))
    if reward is None:
        reward = torch.tensor(float("nan"))
    if terminated is None:
        terminated = torch.tensor(float("nan"))
    return TensorDict(
        obs=obs,
        action=action.unsqueeze(0),
        reward=reward.unsqueeze(0),
        terminated=terminated.unsqueeze(0),
        batch_size=(1,),
    )


def train_tdmpc2(
    seed: int,
    total_timesteps: int = TOTAL_TIMESTEPS,
    env_id: str = TRAIN_ENV_ID,
    model_size: int = 1,
    use_wandb: bool = True,
    log_every: int = 1000,
) -> Path:
    """Train one TD-MPC2 seed end-to-end. Returns final checkpoint path."""
    np.random.seed(seed); torch.manual_seed(seed)
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

    run_name = f"tdmpc2_{env_id}_seed{seed}"
    ckpt_dir = CHECKPOINTS_DIR / run_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    if use_wandb:
        wandb.init(
            project=WANDB_PROJECT, entity=WANDB_ENTITY, name=run_name,
            config={"algo": "TD-MPC2", "env_id": env_id, "seed": seed,
                    "total_timesteps": total_timesteps, "model_size": model_size},
            reinit=True,
        )

    # Build env + cfg + agent + buffer
    env = make_tdmpc2_highway_env(env_id=env_id, seed=seed)
    cfg = _build_cfg(env, seed=seed, total_timesteps=total_timesteps, model_size=model_size)
    agent = TDMPC2(cfg)
    buffer = Buffer(cfg)

    # Online RL loop (mirrors OnlineTrainer.train)
    step = 0
    ep_idx = 0
    done = True
    tds: list = []
    info: dict = {}
    ep_rewards_recent: list[float] = []
    ep_lengths_recent: list[int] = []
    ep_collisions_recent: list[bool] = []
    t_start = time.time()

    while step <= total_timesteps:
        # Reset on episode boundary
        if done:
            if step > 0:
                # Push completed episode into buffer
                episode_reward = float(torch.tensor([t["reward"] for t in tds[1:]]).sum())
                episode_length = len(tds)
                episode_collided = bool(info.get("episode_collision", False))

                ep_rewards_recent.append(episode_reward)
                ep_lengths_recent.append(episode_length)
                ep_collisions_recent.append(episode_collided)

                if use_wandb:
                    wandb.log({
                        "rollout/ep_reward": episode_reward,
                        "rollout/ep_length": episode_length,
                        "metrics/collision": int(episode_collided),
                        "metrics/success":   float(info.get("success", 0.0)),
                        "global_step":       step,
                    }, step=step)

                ep_idx = buffer.add(torch.cat(tds))

            obs = env.reset()
            tds = [_to_td(obs, env)]

        # Action: random during seed phase, planned afterwards
        if step > cfg.seed_steps:
            action = agent.act(obs, t0=(len(tds) == 1))
        else:
            action = env.rand_act()

        obs, reward, done, info = env.step(action)
        tds.append(_to_td(obs, env, action, reward, info["terminated"]))

        # Update agent
        if step >= cfg.seed_steps:
            n_updates = cfg.seed_steps if step == cfg.seed_steps else 1
            if step == cfg.seed_steps:
                print(f"[seed_steps={cfg.seed_steps} reached] pretraining agent...")
            for _ in range(n_updates):
                _ = agent.update(buffer)

        # Periodic console summary
        if (step + 1) % log_every == 0 and ep_rewards_recent:
            tail = slice(-20, None)
            elapsed = time.time() - t_start
            print(
                f"[step {step+1:6d}/{total_timesteps}] "
                f"ep_rew(20)={np.mean(ep_rewards_recent[tail]):6.2f} "
                f"ep_len(20)={np.mean(ep_lengths_recent[tail]):5.1f} "
                f"crash%(20)={100*np.mean(ep_collisions_recent[tail]):3.0f} "
                f"sps={step/elapsed:5.1f}"
            )

        step += 1

    # Save checkpoint
    final_path = ckpt_dir / "final.pt"
    agent.save(str(final_path))
    print(f"\\n[TD-MPC2 seed={seed}] saved -> {final_path}")

    if use_wandb and wandb.run is not None:
        wandb.finish()

    return final_path
'''

(ROOT / "src/training/train_tdmpc2.py").write_text(train_tdmpc2_py)
print("[ok] src/training/train_tdmpc2.py rewritten")

[ok] src/training/train_tdmpc2.py rewritten


In [31]:
# T5 (RE-RUN) — Tiny TD-MPC2 run with the corrected adapter + cfg.
# 3000 steps: enough to cover seed_steps (1000) + start of planning phase.
import importlib
import src.envs.tdmpc2_adapter as a; importlib.reload(a)
import src.training.train_tdmpc2 as m; importlib.reload(m)
from src.training.train_tdmpc2 import train_tdmpc2

_ = train_tdmpc2(seed=99, total_timesteps=3_000, use_wandb=False, log_every=500)
print("\n[ok] TD-MPC2 pipeline functional end-to-end")

Episode length: 200
Discount factor: 0.975
Buffer capacity: 3,000
Storage required: 0.00 GB
Using CUDA:0 memory for storage.
2026-05-04 17:33:12,035 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([3000]) shape [END]


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

[step    500/3000] ep_rew(20)=  2.66 ep_len(20)=  5.5 crash%(20)=  0 sps= 62.6


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[step   1000/3000] ep_rew(20)=  2.45 ep_len(20)=  5.2 crash%(20)=  0 sps= 64.5
[seed_steps=1000 reached] pretraining agent...


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

[step   2500/3000] ep_rew(20)= 17.04 ep_len(20)= 19.4 crash%(20)= 60 sps= 19.6


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

[step   3000/3000] ep_rew(20)= 13.97 ep_len(20)= 16.4 crash%(20)= 50 sps= 19.0

[TD-MPC2 seed=99] saved -> /content/drive/MyDrive/tdmpc2-highway/checkpoints/tdmpc2_highway-v0_seed99/final.pt

[ok] TD-MPC2 pipeline functional end-to-end


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [32]:
# Suppress the spam-level deprecation warnings before T7 launches.
# These come from jupyter_client / wandb's internal use of datetime.utcnow.
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*datetime.datetime.utcnow.*")
warnings.filterwarnings("ignore", message=".*'reinit'.*")

# Also silence at module-import level for any future imports
import os
os.environ["PYTHONWARNINGS"] = "ignore"
print("[ok] deprecation warnings silenced")

[ok] deprecation warnings silenced


In [33]:
# T6 — Launch driver for TD-MPC2 × 3 seeds. Defining only;
# T7 actually runs it. Separating definition from execution lets
# us re-import without re-launching by accident.
import importlib
import src.envs.tdmpc2_adapter as a; importlib.reload(a)
import src.training.train_tdmpc2 as m; importlib.reload(m)
from src.training.train_tdmpc2 import train_tdmpc2
from src.utils.config import SEEDS, TOTAL_TIMESTEPS

print(f"Plan: TD-MPC2 × {len(SEEDS)} seeds = {SEEDS}")
print(f"      {TOTAL_TIMESTEPS:,} steps each, model_size=1 (≈1M params)")
print(f"      Estimated wall-clock on H100: ~75-90 min/seed × 3 ≈ 4 hours total")
print(f"\nReady. Run T7 to launch.")

Plan: TD-MPC2 × 3 seeds = [0, 1, 2]
      100,000 steps each, model_size=1 (≈1M params)
      Estimated wall-clock on H100: ~75-90 min/seed × 3 ≈ 4 hours total

Ready. Run T7 to launch.


In [ ]:
# T7 — 🔴 LONG TRAIN: TD-MPC2 × 3 seeds.
# H100 estimate: ~75-90 min/seed → ~4 hours total.
# Drive checkpoints survive disconnects. wandb logs survive disconnects.
# Don't close this tab — let it run.
tdmpc2_paths = []
for seed in SEEDS:
    print(f"\n{'='*60}\n  TD-MPC2 seed {seed}\n{'='*60}")
    p = train_tdmpc2(
        seed=seed,
        total_timesteps=TOTAL_TIMESTEPS,
        model_size=1,
        use_wandb=True,
        log_every=2_000,
    )
    tdmpc2_paths.append(p)

print("\n" + "="*60)
print("All TD-MPC2 seeds complete:")
for p in tdmpc2_paths:
    print(f"  {p}")
print("="*60)


  TD-MPC2 seed 0


wandb: WARNING Step cannot be set when using tensorboard syncing. Please use `run.define_metric(...)` to define a custom metric to log your step values.


Episode length: 200
Discount factor: 0.975
Buffer capacity: 100,000
Storage required: 0.01 GB
Using CUDA:0 memory for storage.
2026-05-04 17:41:05,467 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([100000]) shape [END]


In [2]:
# Verify all 6 final checkpoints exist on Drive
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/tdmpc2-highway")
ckpts = sorted(ROOT.glob("checkpoints/*/final.*"))
print(f"Found {len(ckpts)} final checkpoints:\n")
for c in ckpts:
    size_mb = c.stat().st_size / 1e6
    print(f"  {c.relative_to(ROOT)}  ({size_mb:.1f} MB)")

Found 8 final checkpoints:

  checkpoints/sac_highway-v0_seed0/final.zip  (3.3 MB)
  checkpoints/sac_highway-v0_seed1/final.zip  (3.3 MB)
  checkpoints/sac_highway-v0_seed2/final.zip  (3.3 MB)
  checkpoints/sac_highway-v0_seed99/final.zip  (3.3 MB)
  checkpoints/tdmpc2_highway-v0_seed0/final.pt  (7.6 MB)
  checkpoints/tdmpc2_highway-v0_seed1/final.pt  (7.6 MB)
  checkpoints/tdmpc2_highway-v0_seed2/final.pt  (7.6 MB)
  checkpoints/tdmpc2_highway-v0_seed99/final.pt  (7.6 MB)
